# Notebook 11: ROI Analysis & Facial Hemodynamics

## Objectives
- Extract per-ROI rPPG signals (forehead, cheeks, full-face)
- Compute per-ROI SNR and heart rate accuracy
- Compare single-ROI vs multi-ROI fusion strategies
- Visualize ROI heatmaps


In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
from src.config import ARCHIVE_DIR, FIGURES_DIR
from src.data.archive_scanner import scan_archive
from scipy import signal as scipy_signal


In [ ]:
# Load per-ROI rPPG traces from video
from src.video.rppg_extended import extract_multi_roi_rgb_traces, compute_per_roi_snr

records = scan_archive(ARCHIVE_DIR)
video_records = [r for r in records if r.video_path is not None]
print(f'Found {len(video_records)} trials with video')


In [ ]:
# Extract multi-ROI traces from first video trial
if video_records:
    rec = video_records[0]
    print(f'Processing {rec.subject_id}/{rec.trial_id}...')
    df = extract_multi_roi_rgb_traces(rec.video_path, max_frames=300)
    print(f'Extracted {len(df)} frames, {len(df.columns)} signals')
    df.head(3)
else:
    print('No video records found')


In [ ]:
# Compute per-ROI SNR for all rPPG methods
if video_records:
    fps = 30.0
    snr_results = compute_per_roi_snr(df, fps)
    snr_df = pd.DataFrame(snr_results).T
    snr_df = snr_df.round(2)
    print('Per-ROI / Per-Method SNR (dB):')
    print('=' * 60)
    print(snr_df.to_string())

    # Visualize SNR comparison
    if len(snr_df) > 0:
        fig, ax = plt.subplots(figsize=(12, 5))
        methods = sorted(set('_'.join(c.split('_')[:-2]) for c in snr_df.index if 'forehead' in c))
        rois = ['forehead', 'left_cheek', 'right_cheek']
        x = np.arange(len(rois))
        width = 0.2
        for i, method in enumerate(['rppg_green', 'rppg_chrom', 'rppg_pos', 'rppg_ica'][:3]):
            vals = []
            for roi in rois:
                col = f'{roi}_{method}'
                if col in snr_df.index:
                    vals.append(snr_df.loc[col, 'snr_db'])
                else:
                    vals.append(np.nan)
            ax.bar(x + i * width, vals, width, label=method)
        ax.set_xticks(x + width)
        ax.set_xticklabels(rois)
        ax.set_ylabel('SNR (dB)')
        ax.set_title('Per-ROI rPPG SNR Comparison')
        ax.legend()
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'roi_snr_comparison.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print('No video data to analyze')


In [ ]:
# Multi-ROI Fusion: Compare fusion strategies
if video_records:
    from sklearn.metrics import mean_absolute_error

    # Load ground-truth HR from aligned data
    from src.config import ALIGNED_DIR
    aligned_path = ALIGNED_DIR / f'{rec.subject_id}_{rec.trial_id}_aligned_1hz.csv'
    aligned = pd.read_csv(aligned_path) if aligned_path.exists() else None
    gt_hr = aligned['hr'].mean() if aligned is not None else 75.0

    # Compute HR from each ROI and each method
    fps = 30.0
    hr_estimates = {}
    for col in df.columns:
        if not any(col.endswith(m) for m in ['rppg_green', 'rppg_chrom', 'rppg_pos', 'rppg_ica']):
            continue
        sig = df[col].dropna().to_numpy()
        if len(sig) < 12:
            continue
        freqs, psd = scipy_signal.welch(sig, fs=fps, nperseg=min(256, len(sig)))
        hr_band = (freqs >= 0.7) & (freqs <= 4.0)
        if not hr_band.any():
            continue
        peak_hz = freqs[hr_band][np.argmax(psd[hr_band])]
        hr_estimates[col] = peak_hz * 60

    # Create comparison table
    if hr_estimates:
        hr_df = pd.DataFrame(list(hr_estimates.items()), columns=['Signal', 'HR_BPM'])
        hr_df['ROI'] = hr_df['Signal'].apply(lambda x: x.split('_')[0])
        hr_df['Method'] = hr_df['Signal'].apply(lambda x: '_'.join(x.split('_')[1:4]))
        hr_df['Error'] = abs(hr_df['HR_BPM'] - gt_hr)
        print(f'Ground Truth HR: {gt_hr:.1f} BPM')
        print('=' * 70)
        print(hr_df.to_string(index=False))

        # Best ROI for each method
        best = hr_df.loc[hr_df.groupby('Method')['Error'].idxmin()]
        print('\nBest ROI per method:')
        print(best[['Method', 'ROI', 'HR_BPM', 'Error']].to_string(index=False))


In [ ]:
# ROI Fusion Strategies
if video_records:
    from src.video.rppg import green_rppg, chrom_rppg, pos_rppg

    # Strategy 1: Average all ROIs (full-face approach)
    full_face_cols = [c for c in df.columns if '_rppg_green' in c]
    if full_face_cols and len(full_face_cols) >= 2:
        signals = [df[c].to_numpy() for c in full_face_cols]
        avg_signal = np.nanmean(signals, axis=0)
        freqs, psd = scipy_signal.welch(avg_signal, fs=fps, nperseg=min(256, len(avg_signal)))
        hr_band = (freqs >= 0.7) & (freqs <= 4.0)
        if hr_band.any():
            hr_fusion_avg = freqs[hr_band][np.argmax(psd[hr_band])] * 60
            print(f'Fusion (Average): {hr_fusion_avg:.1f} BPM, Error: {abs(hr_fusion_avg - gt_hr):.1f} BPM')

    # Strategy 2: SNR-weighted fusion
    weighted_signal = np.zeros(len(list(signals)[0]))
    total_weight = 0
    for col in full_face_cols:
        sig = df[col].to_numpy()
        snr_val = np.nanvar(sig) / (np.nanvar(np.diff(sig)) + 1e-12)
        weight = min(snr_val, 100)
        weighted_signal += weight * sig
        total_weight += weight
    if total_weight > 0:
        weighted_signal /= total_weight
        freqs, psd = scipy_signal.welch(weighted_signal, fs=fps, nperseg=min(256, len(weighted_signal)))
        hr_band = (freqs >= 0.7) & (freqs <= 4.0)
        if hr_band.any():
            hr_fusion_weighted = freqs[hr_band][np.argmax(psd[hr_band])] * 60
            print(f'Fusion (SNR-Weighted): {hr_fusion_weighted:.1f} BPM, Error: {abs(hr_fusion_weighted - gt_hr):.1f} BPM')

    # Strategy 3: Best single ROI (minimum error)
    if len(hr_df) > 0:
        best_roi = hr_df.loc[hr_df['Error'].idxmin()]
        best_roi_name = best_roi["ROI"]
        best_method = best_roi["Method"]
        best_hr = best_roi["HR_BPM"]
        best_err = best_roi["Error"]
        print(f'Best Single ROI: {best_roi_name} ({best_method}), HR: {best_hr:.1f} BPM, Error: {best_err:.1f} BPM')


In [ ]:
# ROI Heatmap Visualization
if video_records:
# Create a simple face ROI heatmap
    roi_snrs = {}
    for roi in ['forehead', 'left_cheek', 'right_cheek']:
        col = f'{roi}_rppg_chrom'
        if col in snr_df.index:
            roi_snrs[roi] = snr_df.loc[col, 'snr_db']

    if roi_snrs:
        fig, ax = plt.subplots(figsize=(8, 6))
        # Schematic face plot with SNR values
        face_coords = {
            'forehead': (0.5, 0.75),
            'left_cheek': (0.25, 0.4),
            'right_cheek': (0.75, 0.4),
        }
        for region, (x, y) in face_coords.items():
            if region in roi_snrs:
                color = 'red' if roi_snrs[region] < 0 else 'orange' if roi_snrs[region] < 3 else 'green'
                ax.scatter(x, y, s=2000, c=color, alpha=0.5, edgecolors='black', linewidth=2)
                ax.text(x, y, f'{region}\n{roi_snrs[region]:.1f} dB', ha='center', va='center', fontsize=9, fontweight='bold')
        # Draw face outline
        face = plt.Circle((0.5, 0.5), 0.35, fill=False, edgecolor='black', linewidth=2)
        ax.add_patch(face)
        eyes = [plt.Circle((0.35, 0.65), 0.05, fill=False, edgecolor='gray'),
                plt.Circle((0.65, 0.65), 0.05, fill=False, edgecolor='gray')]
        for eye in eyes: ax.add_patch(eye)
        ax.set_xlim(0, 1), ax.set_ylim(0, 1)
        ax.set_aspect('equal'), ax.axis('off')
        ax.set_title('Facial ROI SNR Heatmap (CHROM Method)', fontsize=14)
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'roi_heatmap.png', dpi=150, bbox_inches='tight')
        plt.show()


In [ ]:
print('ROI Analysis complete.')
